In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

#PATHS
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

In [ ]:
# IDENTIFYING SCHEMA AND MISSING DATA
txn=pd.read_csv(RAW / "transactions.csv")
print(txn.shape)   #rows and columns
print(txn.dtypes)  #column data types
print(txn.head())  #first 5 rows
print(txn.isnull().sum())  #null count per column

In [ ]:
# IDENTIFYING DUPLICATE ROWS
print("Duplicate Rows: ", txn.duplicated().sum())
print("Duplicate Txns: ", txn.duplicated(subset=["SKU_ID","Date"]).sum())

In [ ]:
#STANDARDISE DATE
txn["Date"] = pd.to_datetime(txn["Date"], format='ISO8601').dt.strftime("%Y-%m-%d")
print(txn["Date"].head())
print(txn["Date"].dtype)

In [ ]:
# VALIDATING SELLING PRICE >= COST PRICE
master = pd.read_csv(RAW / "product_master.csv")

txn = txn.merge(master[["SKU_ID", "Base_Cost_INR"]], on="SKU_ID", how="left")

txn["price_violation"] = txn["Effective_Price_INR"] < txn["Base_Cost_INR"]
print("Price violations:", txn["price_violation"].sum())

violations = txn[txn["price_violation"]]
print(violations[["SKU_ID", "Effective_Price_INR", "Base_Cost_INR"]].head(20))

In [ ]:
## CLEANED-TRANSACTIONS SAVED AS PARQUET
txn_clean = txn.drop_duplicates(subset=["SKU_ID", "Date"]) #0 duplicates, but defensive approach

# Ensuring the shape passed is clean. (Defensive approach)
print("Clean shape:", txn_clean.shape)
print("Price violations retained:", txn_clean["price_violation"].sum())

#Save to the processed folder
txn_clean.to_parquet(PROCESSED / "transactions_clean.parquet", index=False)
print("Saved: transactions_clean.parquet")

In [ ]:
## LOAD AND PROFILE COMPTETITOR_PRICES.CSV
comp = pd.read_csv(RAW / "competitor_prices.csv")

print("Shape:", comp.shape)
print("\nColumn names:")
print(comp.columns.tolist())
print("\nData types:")
print(comp.dtypes)
print("\nFirst 5 rows:")
print(comp.head())
print("\nNull counts:")
print(comp.isnull().sum())

In [ ]:
# IDENTIFYING DUPLICATE ROWS
print("Duplicate Rows: ", comp.duplicated().sum())
print("Duplicate Entries: ", comp.duplicated(subset=["SKU_ID","Competitor_Name","Week_Start_Date"]).sum())

In [ ]:
## COMPARING THE PRODUCT CATALOGUE IN OUR AND THE COMPETITOR's COMPANY
comp_skus=set(comp["SKU_ID"].unique()) #competitor's catalogue
master_skus=set(master["SKU_ID"].unique()) #master is the catalogue of our company

print("Products in comp's catalogue but not in ours:",comp_skus-master_skus)
print("Products in our catalogue but not in comp's:", master_skus-comp_skus)
print("Products in both companies catalogue:", len(comp_skus & master_skus))

In [ ]:
## STANDARDISING THE DATES IN COMPETITOR_PRICES.CSV
comp["Week_Start_Date"]=pd.to_datetime(comp["Week_Start_Date"], format='ISO8601').dt.strftime("%Y-%m-%d")

print("Date sample after standardisation:")
print(comp["Week_Start_Date"].head())

In [ ]:
## PROFILING PRODUCT_MASTER.CSV

print("Shape:", master.shape)
print("\nColumn names:")
print(master.columns.tolist())
print("\nData types:")
print(master.dtypes)
print("\nFirst 5 rows:")
print(master.head())
print("\nNull counts:")
print(master.isnull().sum())

In [ ]:
## CHECKING THE DATA QUALITY OF PRODUCT_MASTER.CSV
print ("Total rows in master:", len(master))
print ("Unique SKUs in master:", master["SKU_ID"].nunique())
print ("Duplicate SKU rows:", master.duplicated(subset =["SKU_ID"]).sum())

In [ ]:
## VERIFYING ALL TRANSACTION SKUS EXIST IN MASTER
txn_skus = set(txn_clean["SKU_ID"].unique())
master_skus = set(master["SKU_ID"].unique())
missing_from_master = txn_skus - master_skus
print("Transaction SKUs missing from master:", missing_from_master)
print("Count missing:", len(missing_from_master))

In [ ]:
#Ensuring that the margin pct is in decimal form
print("Margin_Target_Pct sample:")
print(master["Margin_Target_Pct"].describe())

In [ ]:
#Rename for convenience
master=master.rename(columns={"Margin_Target_Pct":"margin_pct"})

In [ ]:
#Verify Cost Price makes sense
print("\nBase_Cost_INR stats:")
print(master["Base_Cost_INR"].describe())

In [ ]:
master.to_parquet(PROCESSED / "product_master_clean.parquet", index=False)
print("Saved: product_master_clean.parquet")
print("Columns saved:",master.columns.tolist())